In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [3]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [4]:
question = 'I just discovered the course, can i join the course?'
answer = llm(question)
print(answer)

Absolutely—you can ask to join, but it depends on the course’s enrollment policy and whether there are any spots left.

A good next step is to contact the course organizer or registrar and say something like:

> Hi, I just discovered this course and I’m very interested in joining. Could you please let me know if late enrollment is still possible?

If you want, I can help you write a more polished message.


In [5]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [6]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [7]:
print(prompt)


Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
I just discovered the course, can i join the course?

Context:

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students 

In [8]:
answer = llm(prompt)
print(answer)

Yes, you can still join the course. If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [9]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [1]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()


In [11]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 471},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 144},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 253}]

In [5]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1406

In [13]:
documents[1405]

{'id': '0d200c8c58',
 'course': 'mlops-zoomcamp',
 'section': 'Capstone Project',
 'question': 'Homework: What is the criteria of scoring home work?',
 'answer': 'Each homework assignment has a scoring system based on the following criteria:\n\n- Answering 6 questions correctly: **6 points**\n- Adding 7 public learning items: **7 points**\n- Adding 1 valid question to the FAQ: **1 point**\n\nIn total, you can earn up to **14 points** per homework, which will contribute to the leaderboard ranking.'}

In [6]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [10]:
index.docs

[{'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: When does the course start?',
  'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."},
 {'id': 'bfafa427b3',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: What are the prerequisites for this course?',
  'answer': "To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful 

In [8]:
index.text_matrices

{'question': <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 14120 stored elements and shape (1406, 2695)>,
 'section': <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 4903 stored elements and shape (1406, 103)>,
 'answer': <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 74353 stored elements and shape (1406, 8191)>}

In [9]:
index.keyword_df

,course
0,data-engineering-zoomcamp
1,data-engineering-zoomcamp
2,data-engineering-zoomcamp
3,data-engineering-zoomcamp
4,data-engineering-zoomcamp
...,...
1401,mlops-zoomcamp
1402,mlops-zoomcamp
1403,mlops-zoomcamp
1404,mlops-zoomcamp


In [15]:
search_results = index.search(
    question,
    boost_dict={'question':2.0, 'section': 1.0, 'answer': 1.0}, 
    filter_dict={'course': 'llm-zoomcamp'}, 
    num_results=5)


search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not state a total number of hours.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge i

In [16]:
def search(question, course='llm-zoomcamp'):
    boost_dict={'question':2.0, 'section': 0.5, 'answer': 1.0} 
    filter_dict={'course': course}


    return index.search(
        question, 
        boost_dict=boost_dict, 
        filter_dict=filter_dict, 
        num_results=5
    )

In [17]:
search_results = search(question)

In [18]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not state a total number of hours.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge i

In [19]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [20]:
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

In [21]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [22]:
context = build_context(search_results)
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: How should I start the course and follow the weekly workflow?
A: Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalk

In [23]:
USER_PROMPT_TEMPLATE.format(question=question, context=context)

'\nQuestion:\nI just discovered the course, can i join the course?\n\nContext:\nGeneral Course-Related Questions\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ: Does the course certificate show the number of course hours?\nA: No. The certificate does not state a total number of hours.\n\nGeneral Course-Related Questions\nQ: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nA: You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nGeneral Course-Related Questions\nQ: How should I start the course and follow the weekly workflow?\nA: Start with the [LLM Zoomcamp docs](https://da

In [24]:
def build_prompt(questions, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question, 
        context=context
    )
    return prompt.strip()

In [25]:
prompt = build_prompt(question, search_results)

In [26]:
print(prompt)

Question:
I just discovered the course, can i join the course?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: How should I start the course and follow the weekly workflow?
A: Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/cours

In [27]:
response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )


In [28]:
response.output_text

'Yes, you can still join the course.\n\nIf you want a certificate, make sure to submit your project while the course is still accepting submissions.'

In [29]:
response.usage

ResponseUsage(input_tokens=495, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=33, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=528)

In [30]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00051975

this  amount od dollar was spent


In [31]:
message_history = [
    {'role': 'developer', 'content': INSTRUCTIONS},
    {'role': 'user', 'content': prompt}
]
response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=message_history
    )

In [32]:
response.output_text

'Yes, you can still join the course. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [33]:
def llm(instructions, user_prompt, model='gpt-5.4-mini'):

    message_history = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]
    response = openai_client.responses.create(
        model=model,
        input=message_history
    )
    return response.output_text

In [34]:
def rag(query, model='gpt-5.4-mini'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [35]:
answer = rag(answer)
print(answer)

Yes, you can still join the course. If you want a certificate, you need to submit your project while submissions are still being accepted.
